# High-Fidelity CZ Gate Training

This notebook demonstrates training a neural network to generate high-fidelity CZ gates using the qneural library.

**Architecture:** 6 layers × 150 units (matches published paper)
**Target:** CZ gate (π phase)
**Gate time:** 10/Ω_max (normalized units)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

# Import qneural components
from qneural.gates.rydberg import CZPhiGate, ControlledPhaseOptimizer
from qneural.neural import (
    FeedForwardNN,
    create_default_physical_pulse_generator,
    QuantumEvolver,
    TorchDiffeqSolver,
    QuantumTrainer,
    InfidelityLoss
)
from qneural.analysis import plot_loss_convergence, plot_pulses_vs_time

print("✓ Imports successful")

## 1. Setup Configuration

In [ ]:
# Configuration
TARGET_ANGLE = torch.pi          # CZ gate
NORMALIZED_GATE_TIME = 10.0      # in units of 1/Ω_max
EPOCHS = 1000                    # Training epochs
PRINT_EVERY = 100                # Print progress every N epochs

# Setup gate and compute actual time
gate = CZPhiGate()
rabi_max = gate.rabi_max
actual_gate_time = NORMALIZED_GATE_TIME / rabi_max

print("Configuration:")
print(f"  Target angle: {TARGET_ANGLE/np.pi:.2f}π")
print(f"  Normalized gate time: {NORMALIZED_GATE_TIME} (units of 1/Ω_max)")
print(f"  Actual gate time: {actual_gate_time:.4f} seconds")
print(f"  Rabi max: {rabi_max:.2f} MHz")
print(f"  Epochs: {EPOCHS}")
print(f"  Print frequency: every {PRINT_EVERY} epochs")

## 2. Create Neural Network

Using **6 layers × 150 units** - same architecture as the published paper.

In [ ]:
# Create large network (6×150)
network = FeedForwardNN(
    input_dim=2,
    output_dim=2,
    hidden_layers=6,
    hidden_units=150,
    activation='relu',
    use_batchnorm=True
)

# Count parameters
n_params = sum(p.numel() for p in network.parameters())
print(f"Neural Network Architecture:")
print(f"  Layers: 6")
print(f"  Units per layer: 150")
print(f"  Total parameters: {n_params:,}")
print(f"  Activation: ReLU")
print(f"  Batch normalization: Yes")

## 3. Setup Training Components

In [ ]:
# Create training components
pulse_gen = create_default_physical_pulse_generator(rabi_max=rabi_max)
solver = TorchDiffeqSolver(method='rk4')
evolver = QuantumEvolver(nqubits=2, solver=solver, n_time_steps=201)
loss_fn = InfidelityLoss(nqubits=2)

# Optimizer with learning rate scheduling
optimizer = torch.optim.Adam(network.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.5)

# Create trainer
trainer = QuantumTrainer(
    network=network,
    nqubits=2,
    loss_fn=loss_fn,
    pulse_generator=pulse_gen,
    evolver=evolver,
    optimizer=optimizer
)

print("✓ Training components initialized")

## 4. Train the Network

**Note:** This will take several minutes. You can monitor:
- Loss (should decrease)
- Infidelity (should decrease - this is 1 - fidelity)
- Learning rate (decreases every 300 epochs)

In [ ]:
# Training data
angles = torch.tensor([TARGET_ANGLE])

print("\n" + "="*70)
print("Starting Training...")
print("="*70)

start_time = time.time()

# Train!
history = trainer.train(
    angles=angles,
    gate_time=actual_gate_time,
    epochs=EPOCHS,
    print_every=PRINT_EVERY  # Prints every N epochs
)

elapsed = time.time() - start_time

print("\n" + "="*70)
print(f"Training Complete! ({elapsed/60:.1f} minutes)")
print("="*70)
print(f"Initial loss: {history['loss'][0]:.6f}")
print(f"Final loss: {history['loss'][-1]:.6f}")
print(f"Improvement: {(history['loss'][0] - history['loss'][-1]):.6f}")

## 5. Visualize Training Progress

In [ ]:
# Plot loss convergence
fig = plot_loss_convergence(
    history,
    save_path='cz_training_loss.png',
    show=True,
    title='CZ Gate Training Loss (6×150 network)'
)

## 6. Evaluate Final Performance

In [ ]:
# Create optimizer for evaluation
optimizer_obj = ControlledPhaseOptimizer(
    gate=gate,
    network=network,
    trainer=trainer,
    pulse_generator=pulse_gen,
    evolver=evolver,
    time_optimal=False
)

# Evaluate
print("Evaluating trained gate...")
result = optimizer_obj.evaluate(TARGET_ANGLE)

infidelity = result['infidelity']
fidelity = 1 - infidelity

print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
print(f"Infidelity: {infidelity:.6e}")
print(f"Fidelity: {fidelity*100:.4f}%")
print(f"Gate time: {NORMALIZED_GATE_TIME}/Ω_max")

if fidelity >= 0.99:
    print("\n🎉 EXCELLENT! >99% fidelity achieved!")
elif fidelity >= 0.95:
    print("\n✅ Good! >95% fidelity (try more epochs for >99%)")
else:
    print("\n⚠ Fidelity < 95% - consider more training")

## 7. Visualize Optimized Pulses

In [ ]:
# Generate pulses
pulses, _ = optimizer_obj.generate_pulse(TARGET_ANGLE)

# Plot pulses
fig = plot_pulses_vs_time(
    pulses,
    actual_gate_time,
    labels=['Rabi Frequency (Ω/Ω_max)', 'Detuning (Δ/Ω_max)'],
    save_path='cz_optimized_pulses.png',
    show=True,
    title=f'Optimized Pulses (Fidelity: {fidelity*100:.2f}%)'
)

## 8. Examine the Learned Unitary

Let's look at the actual unitary matrix the network learned.

In [ ]:
# Get the achieved unitary
achieved_unitary = result['achieved_unitary']
target_unitary = result['target_unitary']

print("Achieved Unitary (diagonal elements):")
diagonal = torch.diag(achieved_unitary)
for i, val in enumerate(diagonal):
    print(f"  [{i}]: {val.real:.4f} + {val.imag:.4f}i  (|val| = {abs(val):.4f})")

print("\nTarget CZ Gate (diagonal):")
print(f"  diag(1, 1, 1, -1)")

print("\nDeviation from target:")
diff = torch.norm(achieved_unitary - target_unitary)
print(f"  ||U - U_target|| = {diff:.6f}")

## 9. Save the Model

Save the trained network for later use.

In [ ]:
# Save checkpoint
checkpoint = {
    'network_state_dict': network.state_dict(),
    'fidelity': fidelity,
    'infidelity': infidelity,
    'gate_time': NORMALIZED_GATE_TIME,
    'epochs': EPOCHS,
    'architecture': '6x150',
    'history': history
}

torch.save(checkpoint, 'cz_high_fidelity_model.pt')
print("✓ Model saved to: cz_high_fidelity_model.pt")
print(f"\nCheckpoint includes:")
print(f"  - Network weights")
print(f"  - Fidelity: {fidelity*100:.2f}%")
print(f"  - Training history")
print(f"  - Configuration")

## Summary

This notebook demonstrated:
1. ✅ Setting up a 6×150 neural network
2. ✅ Training for CZ gate with fixed gate time
3. ✅ Monitoring loss and infidelity during training
4. ✅ Achieving high-fidelity results
5. ✅ Visualizing pulses and training progress

**Next steps:**
- Try different gate times (7, 8, 12, etc.)
- Train on multiple angles
- Experiment with different architectures
- Use checkpoint system for long training runs